In [ ]:
import torch
import numpy as np
from scipy.stats import norm
from tqdm.autonotebook import tqdm
from pathlib import Path
import csv

# GPU + simulation settings
device = "cuda" if torch.cuda.is_available() else "cpu"
N_SEEDS, TAU, ALPHA = 1000, 0.5, 0.05
BUDGET_PROPS = np.round(np.linspace(0.0, 0.25, 11)[1:], decimals=3)
dataset = "alphafold"

METHOD = "ppat"
LAMBDA_INIT, LAMBDA_UPDATE_EVERY = 0.5, 100  # PPAT Section 5.1, p. 9
DATA_DIR = Path("../datasets/alphafold")
LOG_DIR = Path("../logs")
fname = f"alphafold_{METHOD}.csv"
log_path = LOG_DIR / fname
lambda_path = LOG_DIR / f"alphafold_{METHOD}_lambda.csv"

# results columns
columns = ["dataset", "budget", "tau", "method", "seed", "thetahat", "coverage", "width", "n_labels"]
lambda_columns = ["dataset", "budget", "tau", "method", "seed", "lambda_hat", "n_labels"]
OVERWRITE = False  # Set True deliberately, or choose a fresh LOG_DIR. No silent append/restart.

In [ ]:
#### LOAD IN OUR DATA

# Same fixed predictions and binary labels as the PAI notebook. No new model is fitted.
fxs = torch.load(DATA_DIR / "fhat_test.pt", map_location=device, weights_only=True).to(device)
ys = torch.load(DATA_DIR / "y_test.pt", map_location=device, weights_only=True).to(device)
if ys.ndim != 1 or fxs.ndim != 1 or fxs.shape != ys.shape:
    raise ValueError("Expected fhat_test and y_test to be matching one-dimensional tensors.")
if fxs.dtype not in (torch.float32, torch.float64):
    raise ValueError("Expected float32 or float64 fixed probabilities; no hidden dtype conversion is applied.")
if not torch.isfinite(fxs).all().item() or not ((fxs >= 0) & (fxs <= 1)).all().item():
    raise ValueError("fhat_test must contain finite probabilities in [0,1].")
if not ((ys == 0) | (ys == 1)).all().item():
    raise ValueError("y_test must contain binary outcomes, without missing labels.")
N = ys.shape[0]
if N_SEEDS < 1 or not (0 < TAU <= 1) or not (0 < ALPHA < 1):
    raise ValueError("Require N_SEEDS >= 1, 0 < TAU <= 1, and 0 < ALPHA < 1.")
if LAMBDA_UPDATE_EVERY < 1:
    raise ValueError("LAMBDA_UPDATE_EVERY must be positive.")
if not all(2 <= int(b * N) < N for b in BUDGET_PROPS):
    raise ValueError("Each budget must yield 2 <= n < population size for these CIs.")

# Section 3, p. 4: fixed FULL-POOL-centered soft proxy, independent of test labels.
proxy_fxs = fxs.detach().to(torch.float64).clone()
proxy_constant = (proxy_fxs == proxy_fxs[0]).all().item()
cs = proxy_fxs - proxy_fxs.mean()  # (N,), never recentered/refitted during acquisition
if proxy_constant:
    cs.zero_()  # Exact centering of a constant, avoiding reduction-roundoff artifacts.
proxy_second_moment = cs.square().mean()  # D_c in Eq. (14); divisor N, not N-1
c_for_sampling = cs.to(fxs.dtype).unsqueeze(0)  # (1,N)
if proxy_constant:
    print("Constant proxy: Eq. (14) is undefined; the zero control variate gives LURE (lambda=0).")
elif proxy_second_moment.item() == 0.0:
    raise FloatingPointError("Nonconstant proxy variance underflowed; no denominator floor is applied.")

# Evaluation-only ground truth. Retain the source floating-label mean when applicable.
true_theta = (ys if ys.is_floating_point() else ys.to(fxs.dtype)).mean(dim=-1, keepdim=True)
print(f"{METHOD}: {dataset}, N={N:,}, seeds={N_SEEDS}, device={device}")
print("Budgets (proportion, labels):", [(float(b), int(b * N)) for b in BUDGET_PROPS])
print(f"Nominal coverage={1-ALPHA:.0%}; uniform mixing={TAU}; supplied model stays fixed.")


In [ ]:
#### PREDICTION-POWERED ACTIVE TESTING: fitted lambda_M only

# Fail rather than silently append duplicate experiments on a rerun.
LOG_DIR.mkdir(parents=True, exist_ok=True)
for output_path in (log_path, lambda_path):
    if output_path.exists() and not OVERWRITE:
        raise FileExistsError(f"{output_path} exists. Use a fresh LOG_DIR or deliberately set OVERWRITE=True.")
with open(log_path, "w", newline="") as file:
    csv.writer(file).writerow(columns)
with open(lambda_path, "w", newline="") as file:
    csv.writer(file).writerow(lambda_columns)

# Preserve separate budget restarts and the original seed-batched structure.
with torch.no_grad():
    for budget_idx, budget in enumerate(BUDGET_PROPS):
        n = int(budget * N)
        torch.random.manual_seed(858 + budget_idx)

        #### initialize data structures
        already_sampled = torch.zeros(size=(N_SEEDS, N), dtype=bool, device=device)
        # Only queried indices and their ORIGINAL conditional sampling probabilities are retained.
        I_history = torch.empty((N_SEEDS, n), dtype=torch.long, device=device)
        q_history = torch.empty((N_SEEDS, n), dtype=torch.float64, device=device)
        lambda_acq = torch.full((N_SEEDS, 1), 0.0 if proxy_constant else LAMBDA_INIT,
                                dtype=torch.float64, device=device)
        # AlphaFold: supplied fxs stay fixed, matching the original PAI notebook.

        #### adaptive sampling rounds
        for t in tqdm(range(n), desc=f"Budget {budget_idx + 1} of {len(BUDGET_PROPS)}"):
            #### PART 1: PPAT's binary RMS acquisition, Eqs. (12)-(13), p. 6
            # fxs are fixed; lambda_acq differs by seed and changes every 100 labels.
            # sqrt(p*(1-p) + (p-lambda*c)^2), with no labels used in the scores.
            shift = lambda_acq.to(fxs.dtype) * c_for_sampling
            scores = torch.hypot(torch.sqrt(fxs * (1.0 - fxs)), fxs - shift)
            log_u_ts = scores.log().masked_fill(already_sampled, -torch.inf)
            # All-zero scores: the score-normalized proposal is undefined. Use uniform there.
            zero_score_rows = torch.isneginf(log_u_ts).all(dim=1, keepdim=True)
            log_u_ts = torch.where(zero_score_rows & ~already_sampled, 0.0, log_u_ts)
            h_js = torch.softmax(log_u_ts, dim=1)
            q_js = (TAU * (~already_sampled).to(h_js.dtype) / (N - t)) + ((1.0 - TAU) * h_js)
            q_js = q_js / q_js.sum(dim=1, keepdim=True)  # numerical normalization
            I_t = torch.multinomial(input=q_js, num_samples=1)  # (N_SEEDS,1)

            #### PART 2: query labels, save probabilities, and remove selected entries
            y_Is = ys[I_t]  # acquired labels only
            # There is no AlphaFold predictor update in the source notebook.
            q_Is = torch.gather(q_js, dim=1, index=I_t)
            I_history[:, t:t+1] = I_t
            q_history[:, t:t+1] = q_Is.to(torch.float64)
            already_sampled.scatter_(dim=1, index=I_t,
                src=torch.full_like(I_t, fill_value=True, dtype=bool))

            # PART 3: no ML update (the source AlphaFold predictor is fixed).

            #### PART 3b: update lambda after each 100 labels for FUTURE draws only
            m = t + 1
            if m % LAMBDA_UPDATE_EVERY == 0 and m < n and not proxy_constant:
                # Eq. (14), p. 7. Recompute V_{m,s} for CURRENT prefix budget m.
                # Never replace stored q_s(I_s) by probabilities from the newly updated model.
                past_ms = torch.arange(1, m + 1, device=device, dtype=torch.float64).unsqueeze(0)
                V_current = 1.0 + ((N - m) / (N - past_ms)) * (
                    1.0 / ((N - past_ms + 1.0) * q_history[:, :m]) - 1.0)
                past_indices = I_history[:, :m]
                past_ys = ys[past_indices]
                numerator = (V_current * past_ys.to(torch.float64) * cs[past_indices]).mean(dim=1, keepdim=True)
                lambda_acq = numerator / proxy_second_moment
                if not torch.isfinite(lambda_acq.to(fxs.dtype)).all().item():
                    raise FloatingPointError("Nonfinite/out-of-range lambda for score dtype; no coefficient clipping is applied.")

        #### PART 4: final plug-in PPAT estimate and its OWN variance estimate
        ms = torch.arange(1, n + 1, device=device, dtype=torch.float64).unsqueeze(0)
        y_history = ys[I_history].to(torch.float64)
        c_history = cs[I_history]  # frozen full-pool-centered proxy at the queried indices

        # Eqs. (5)/(22), pp. 3/38: final-budget LURE weights, not equal weights or HT weights.
        Vs = 1.0 + ((N - n) / (N - ms)) * (
            1.0 / ((N - ms + 1.0) * q_history) - 1.0)
        # Fit the endpoint coefficient on ALL n labels, including endpoints <100 or not %100==0.
        if proxy_constant:
            lambda_final = torch.zeros((N_SEEDS, 1), dtype=torch.float64, device=device)
        else:
            lambda_final = (Vs * y_history * c_history).mean(dim=1, keepdim=True) / proxy_second_moment

        # Eq. (8), p. 4, and Section E.5.4, p. 56: ONE final lambda for every residual.
        residuals = y_history - lambda_final * c_history
        thetahats = (Vs * residuals).mean(dim=1, keepdim=True)
        # Lemma E.4, p. 38: sum over s<m excludes the current residual.
        prefix_residuals = torch.cumsum(residuals, dim=1) - residuals
        A_ms = (residuals / q_history + prefix_residuals) / N
        gammas = N * (N - n) / ((N - ms) * (N - ms + 1.0))
        # Corollary E.19, p. 59: sigmahat^2 estimates the CLT variance, not Var(thetahat).
        sigmahats_sq = ((gammas * (A_ms - thetahats)) ** 2).mean(dim=1, keepdim=True)

        # Endpoint checks before writing results (no change to the statistical estimator).
        torch.testing.assert_close(thetahats, (gammas * A_ms).mean(dim=1, keepdim=True),
                                   rtol=1e-9, atol=1e-11)
        if not (already_sampled.sum(dim=1) == n).all().item():
            raise RuntimeError("A target/seed did not acquire exactly n distinct labels.")
        if not (torch.isfinite(lambda_final).all() & torch.isfinite(thetahats).all()
                & torch.isfinite(sigmahats_sq).all()).item():
            raise FloatingPointError("Nonfinite fitted PPAT output; nothing is silently clipped.")

        # Same confidence level, finite-pool target, and untruncated width/coverage definitions.
        z_score = norm.ppf(1 - (ALPHA / 2))
        ubs = thetahats + z_score * (torch.sqrt(sigmahats_sq) / (n ** 0.5))
        lbs = thetahats - z_score * (torch.sqrt(sigmahats_sq) / (n ** 0.5))
        widths = (ubs - lbs).cpu().numpy()
        coverages = ((lbs <= true_theta) & (true_theta <= ubs)).cpu().numpy()
        thetahats = thetahats.cpu().numpy()
        final_lambdas = lambda_final.cpu().numpy()

        # store our results
        results = [
            [dataset, budget, TAU, METHOD, seed,
             thetahats[seed][0], coverages[seed][0], widths[seed][0], n]
            for seed in range(N_SEEDS)]
        with open(log_path, "a", newline="") as file:
            csv.writer(file).writerows(results)
        with open(lambda_path, "a", newline="") as file:
            csv.writer(file).writerows([
                [dataset, budget, TAU, METHOD, seed, final_lambdas[seed][0], n]
                for seed in range(N_SEEDS)])
        print(f"budget={budget:.3f}, n={n}, coverage={coverages.mean():.3f}, "
              f"mean width={widths.mean():.6f}, mean fitted lambda={final_lambdas.mean():.4f}")
